# 03 — Preprocessing
> Cleans the raw data: handles outliers, imputes missing values, encodes categoricals,
> applies target transforms, and saves a model-ready interim dataset.

**Prerequisite:** `01_data_ingestion.ipynb`

## 0 · Colab Setup

In [ ]:
import sys, os
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !git clone https://github.com/YOUR_USERNAME/tree_carbon_ml.git 2>/dev/null || true
    %cd tree_carbon_ml
    !pip install -r requirements.txt -q
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))
print('Ready')

## 1 · Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import RobustScaler, StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from scipy import stats

pd.set_option('display.float_format', '{:.4f}'.format)
TARGET = 'TPH.gs.dC.dN0.01'

df = pd.read_parquet('data/interim/01_raw_validated.parquet')
print(f'Loaded: {df.shape}')
df.head(3)

## 2 · Define Column Groups

In [ ]:
# Identifier columns (never used as features)
ID_COLS = ['PLT_CN']

# Target columns
TARGET_COLS = ['TPH.gs.dC.dN0.01', 'TPH.s.dC.dN0.01', 'TPH.g.dC.dN0.01',
               'EXPN.ha.TPH.gs.dC.dN0.01', 'EXPN.ha.TPH.s.dC.dN0.01', 'EXPN.ha.TPH.g.dC.dN0.01']
TARGET_COLS = [c for c in TARGET_COLS if c in df.columns]

# Categorical/ecoregion columns
CAT_COLS = ['US_L4CODE','NA_L3CODE','NA_L1CODE','e3','e1','e4',
            'e3.state','e4.state','e1.state']
CAT_COLS = [c for c in CAT_COLS if c in df.columns]

# Numeric feature columns
NUM_COLS = ['LAT','LON','EXPN.ha','eco.EXPN.ha','state.EXPN.ha']
NUM_COLS = [c for c in NUM_COLS if c in df.columns]

# State/county IDs (treat as categoricals)
GEO_ID_COLS = ['STATECD','COUNTYCD']
GEO_ID_COLS = [c for c in GEO_ID_COLS if c in df.columns]

print('ID cols        :', ID_COLS)
print('Target cols    :', TARGET_COLS)
print('Numeric cols   :', NUM_COLS)
print('Categorical cols:', CAT_COLS)
print('Geo ID cols    :', GEO_ID_COLS)

## 3 · Remove Exact Duplicates

In [ ]:
n_before = len(df)
df = df.drop_duplicates(subset=['PLT_CN']).reset_index(drop=True)
print(f'Rows before: {n_before:,}  |  After dedup: {len(df):,}  |  Removed: {n_before-len(df):,}')

## 4 · Outlier Handling
We use **IQR-based winsorization** (capping) rather than dropping rows, to preserve spatial coverage.

In [ ]:
def winsorize_col(series: pd.Series, lower_pct=0.01, upper_pct=0.99) -> pd.Series:
    lo, hi = series.quantile(lower_pct), series.quantile(upper_pct)
    return series.clip(lo, hi)

df_clean = df.copy()

# Winsorize target at 1st–99th percentile
for col in TARGET_COLS:
    before = df_clean[col].describe()
    df_clean[col] = winsorize_col(df_clean[col])
    print(f'{col}: min {before["min"]:.2f} → {df_clean[col].min():.2f} | '
          f'max {before["max"]:.2f} → {df_clean[col].max():.2f}')

# Winsorize numeric features
for col in NUM_COLS:
    df_clean[col] = winsorize_col(df_clean[col])

print('\nWinsorization complete.')

In [ ]:
# Verify coordinate validity after winsorization
print('LAT range after clean:', df_clean['LAT'].min(), '→', df_clean['LAT'].max())
print('LON range after clean:', df_clean['LON'].min(), '→', df_clean['LON'].max())

# Remove any rows where coordinates are clearly invalid (0,0)
invalid_coords = (df_clean['LAT'] == 0) | (df_clean['LON'] == 0)
print(f'Invalid coordinate rows: {invalid_coords.sum()}')
df_clean = df_clean[~invalid_coords].reset_index(drop=True)
print(f'Rows after coord filter: {len(df_clean):,}')

## 5 · Missing Value Imputation

In [ ]:
print('Missing values before imputation:')
miss = df_clean.isna().sum()
display(miss[miss > 0])

In [ ]:
# Numeric: median imputation (robust to skew)
if df_clean[NUM_COLS].isna().any().any():
    num_imputer = SimpleImputer(strategy='median')
    df_clean[NUM_COLS] = num_imputer.fit_transform(df_clean[NUM_COLS])
    print('Numeric imputation done.')

# Categoricals: fill with 'UNKNOWN'
for col in CAT_COLS + GEO_ID_COLS:
    n_miss = df_clean[col].isna().sum()
    if n_miss > 0:
        df_clean[col] = df_clean[col].fillna('UNKNOWN')
        print(f'  {col}: filled {n_miss} nulls with UNKNOWN')

print('\nMissing values after imputation:')
miss_after = df_clean.isna().sum()
print(miss_after[miss_after > 0] if miss_after.any() else '✅ None')

## 6 · Categorical Encoding
Encode ecoregion codes using **Label Encoding** for tree-based models and one-hot for linear models.

In [ ]:
# Label Encoding (for gradient boosting / tree models)
label_encoders = {}
for col in CAT_COLS + GEO_ID_COLS:
    le = LabelEncoder()
    df_clean[f'{col}_enc'] = le.fit_transform(df_clean[col].astype(str))
    label_encoders[col] = le
    print(f'  {col} → {col}_enc  ({len(le.classes_)} classes)')

print('\nLabel encoding done.')

In [ ]:
# One-Hot Encoding for high-level ecoregions (L1 only — low cardinality)
ohe_cols = ['NA_L1CODE']
ohe_cols = [c for c in ohe_cols if c in df_clean.columns]
df_ohe = pd.get_dummies(df_clean[ohe_cols], prefix=ohe_cols, drop_first=True)
df_clean = pd.concat([df_clean, df_ohe], axis=1)
print(f'Added {df_ohe.shape[1]} OHE columns for {ohe_cols}')
print('OHE columns:', df_ohe.columns.tolist())

## 7 · Target Transformations
Create multiple transformed versions of the target for different model types.

In [ ]:
# 1. Signed log transform (handles both positive and negative values)
df_clean['target_log'] = np.sign(df_clean[TARGET]) * np.log1p(np.abs(df_clean[TARGET]))

# 2. Binary classification target (sink vs source)
df_clean['target_binary'] = (df_clean[TARGET] > 0).astype(int)

# 3. Tertile-based multiclass (strong sink / neutral / strong source)
q33, q66 = df_clean[TARGET].quantile(0.33), df_clean[TARGET].quantile(0.66)
df_clean['target_class3'] = pd.cut(
    df_clean[TARGET],
    bins=[-np.inf, q33, q66, np.inf],
    labels=[0, 1, 2]  # 0=source, 1=neutral, 2=sink
).astype(int)

print('Target transform stats:')
for col in ['TPH.gs.dC.dN0.01','target_log','target_binary','target_class3']:
    print(f'  {col}: {df_clean[col].describe().to_dict()}')

In [ ]:
# Visualize transforms
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(df_clean[TARGET].dropna(), bins=80, color='#2196F3', edgecolor='white', lw=0.3)
axes[0].set_title('Original Target'); axes[0].axvline(0,color='red',ls='--')

axes[1].hist(df_clean['target_log'].dropna(), bins=80, color='#9C27B0', edgecolor='white', lw=0.3)
axes[1].set_title('Signed Log Transform'); axes[1].axvline(0,color='red',ls='--')

axes[2].bar(['Source (0)','Sink (1)'],
            df_clean['target_binary'].value_counts().sort_index().values,
            color=['#F44336','#4CAF50'])
axes[2].set_title('Binary Target Distribution')

fig.suptitle('Target Variable Transformations', fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/figures/08_target_transforms.png', dpi=150, bbox_inches='tight')
plt.show()

## 8 · Feature Scaling

In [ ]:
# RobustScaler — preferred for skewed distributions
robust_scaler = RobustScaler()
scale_cols = NUM_COLS.copy()

scaled_arr = robust_scaler.fit_transform(df_clean[scale_cols].fillna(0))
scaled_df = pd.DataFrame(scaled_arr, columns=[f'{c}_scaled' for c in scale_cols])
df_clean = pd.concat([df_clean.reset_index(drop=True), scaled_df.reset_index(drop=True)], axis=1)

print('RobustScaler applied to:', scale_cols)
print('Scaled column sample stats:')
display(scaled_df.describe().round(3))

## 9 · Preprocessing Quality Report

In [ ]:
print('=== Preprocessing Summary ===')
print(f'Final shape      : {df_clean.shape}')
print(f'Remaining nulls  : {df_clean.isna().sum().sum()}')
print(f'Numeric cols     : {df_clean.select_dtypes(include=np.number).shape[1]}')
print(f'Encoded cat cols : {len(label_encoders)}')
print(f'OHE cols added   : {df_ohe.shape[1]}')
print(f'Target transforms: target_log, target_binary, target_class3')

# Skewness before/after log transform
print(f'\nTarget skewness:')
print(f'  Original  : {df_clean[TARGET].skew():.3f}')
print(f'  Log-transf: {df_clean["target_log"].skew():.3f}')

## 10 · Save Preprocessed Dataset

In [ ]:
import os
os.makedirs('data/interim', exist_ok=True)

out_path = 'data/interim/03_preprocessed.parquet'
df_clean.to_parquet(out_path, index=False)
print(f'Saved: {out_path}  shape={df_clean.shape}')

# Also save column manifest
manifest = {
    'id_cols': ID_COLS,
    'target_cols': TARGET_COLS + ['target_log','target_binary','target_class3'],
    'numeric_cols': NUM_COLS,
    'categorical_cols': CAT_COLS + GEO_ID_COLS,
    'encoded_cols': [f'{c}_enc' for c in CAT_COLS + GEO_ID_COLS],
    'scaled_cols': [f'{c}_scaled' for c in scale_cols],
    'ohe_cols': df_ohe.columns.tolist(),
}
import json
with open('data/interim/03_column_manifest.json','w') as f:
    json.dump(manifest, f, indent=2)
print('Column manifest saved to data/interim/03_column_manifest.json')

---
### ✅ Notebook 03 Complete
Next: **04_feature_extraction.ipynb**